# TrustExtract-N: Corpus Exploration & Statistical Analysis

## 📌 Overview & Objective
This notebook performs a comprehensive exploratory data analysis (EDA) of the ingested government notice corpus for **TrustExtract-N**.

### Why dataset exploration is essential:
1. **Understand Document Distribution:** Government notices vary significantly in length, authority, and jurisdiction (Central vs State).
2. **Identify Metadata Gaps:** Assessing missing fields (`issuing_authority`, `issue_date`, `doc_id`) helps design resilient token classification schemas.
3. **Token Length Calibration:** Analyzing character and word count distributions guides chunking and token truncation strategies for multilingual transformer encoders.
4. **Export Clean Statistics:** The notebook saves summary statistics to `data/processed/dataset_statistics.json` for reproducible downstream training.

---

## 1. Setup & Environment Initialization

**Why we use these libraries:**
- `pandas`: For structured data manipulation, aggregation, and tabular operations.
- `matplotlib.pyplot`: For rendering histogram plots and bar charts of document distributions.
- `json` & `os`: For reading JSONL corpus files and exporting `dataset_statistics.json`.

In [ ]:
# Import core analytical libraries
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Configure visualization style
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10
plt.style.use('ggplot')

# Define dataset input paths (support running from root or notebooks folder)
BASE_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
KANOONGPT_FILE = os.path.join(BASE_DIR, "data", "processed", "kanoongpt_selected.jsonl")
MYSCHEME_FILE = os.path.join(BASE_DIR, "data", "processed", "myscheme_selected.jsonl")
STATS_OUTPUT_FILE = os.path.join(BASE_DIR, "data", "processed", "dataset_statistics.json")

print(f"Base Directory: {BASE_DIR}")
print(f"Stats Output Target: {STATS_OUTPUT_FILE}")

## 2. Load Processed Notice Corpora

**What this section does:**
Loads JSONL records from `kanoongpt_selected.jsonl` and `myscheme_selected.jsonl` into a unified Pandas DataFrame.

In [ ]:
records = []

# Load KanoonGPT records
if os.path.exists(KANOONGPT_FILE):
    with open(KANOONGPT_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            item['source_dataset'] = 'KanoonGPT'
            records.append(item)

# Load MyScheme records
if os.path.exists(MYSCHEME_FILE):
    with open(MYSCHEME_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            item['source_dataset'] = 'MyScheme'
            item['document_title'] = item.get('filename')
            records.append(item)

df = pd.DataFrame(records)
print(f"Total Loaded Documents: {len(df)}")
df.head(3)

## 3. Analysis Section 1: Dataset Size & Memory Footprint

**Why:** Computes overall row count, memory usage, and column dimensions.

In [ ]:
total_records = len(df)
memory_kb = df.memory_usage(deep=True).sum() / 1024.0

print(f"Dataset Size Summary:")
print(f" - Total Document Count: {total_records}")
print(f" - Total Columns: {len(df.columns)}")
print(f" - Total Memory Usage: {memory_kb:.2f} KB")

## 4. Analysis Section 2: Document Type Distribution

**Why:** Analyzes the distribution across target classes: Notification, Circular, Order, Guidelines, Scheme, etc.

In [ ]:
doc_type_counts = df['document_type'].value_counts(dropna=False).to_dict()
print("Document Type Counts:", doc_type_counts)

# Plot Document Type Distribution
plt.figure(figsize=(8, 4))
pd.Series(doc_type_counts).plot(kind='bar', color='#06b6d4', edgecolor='black')
plt.title("Document Type Distribution in Corpus")
plt.xlabel("Document Type")
plt.ylabel("Number of Documents")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Analysis Section 3: Jurisdiction Distribution

**Why:** Examines geographic / governmental jurisdiction coverage.

In [ ]:
jurisdiction_counts = df['document_jurisdiction'].value_counts(dropna=False).to_dict()
print("Jurisdiction Distribution:", jurisdiction_counts)

plt.figure(figsize=(8, 4))
pd.Series(jurisdiction_counts).head(10).plot(kind='barh', color='#10b981', edgecolor='black')
plt.title("Top 10 Document Jurisdictions")
plt.xlabel("Count")
plt.tight_layout()
plt.show()

## 6. Analysis Section 4: Missing Values Audit

**Why:** Identifies null or unpopulated metadata fields.

In [ ]:
missing_counts = df.isnull().sum().to_dict()
missing_percentages = (df.isnull().sum() / len(df) * 100).round(2).to_dict()

print("Missing Value Counts:", missing_counts)
print("Missing Percentages (%):", missing_percentages)

## 7. Analysis Section 5: Text Length Distribution

**Why:** Computes character and word count statistics to guide model token length selection.

In [ ]:
# Compute character and word lengths
df['char_length'] = df['text'].astype(str).str.len()
df['word_length'] = df['text'].astype(str).apply(lambda x: len(x.split()))

stats_char = df['char_length'].describe().to_dict()
stats_word = df['word_length'].describe().to_dict()

print("Character Length Stats:", stats_char)
print("Word Length Stats:", stats_word)

# Plot Length Histograms
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['char_length'].plot(kind='hist', bins=20, ax=axes[0], color='#8b5cf6', edgecolor='black')
axes[0].set_title("Character Length Distribution")
axes[0].set_xlabel("Character Count")

df['word_length'].plot(kind='hist', bins=20, ax=axes[1], color='#f59e0b', edgecolor='black')
axes[1].set_title("Word Length Distribution")
axes[1].set_xlabel("Word Count")
plt.tight_layout()
plt.show()

## 8. Analysis Section 6: Most Common Issuing Authorities

**Why:** Ranks top statutory bodies and issuing departments.

In [ ]:
authority_counts = df['issuing_authority'].value_counts(dropna=False).head(10).to_dict()
print("Top Issuing Authorities:", authority_counts)

plt.figure(figsize=(8, 4))
pd.Series(authority_counts).plot(kind='bar', color='#ec4899', edgecolor='black')
plt.title("Top Issuing Authorities")
plt.ylabel("Document Count")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 9. Analysis Section 7 & 8: Short vs Long Document Examples

**Why:** Inspects extreme text lengths to evaluate truncation needs.

In [ ]:
print("--- Short Document Samples ---")
short_docs = df.nsmallest(3, 'char_length')[['document_title', 'char_length', 'text']]
for idx, row in short_docs.iterrows():
    print(f"Title: {row['document_title']} ({row['char_length']} chars)")
    print(f"Snippet: {row['text'][:200]}...\n")

print("\n--- Long Document Samples ---")
long_docs = df.nlargest(3, 'char_length')[['document_title', 'char_length', 'text']]
for idx, row in long_docs.iterrows():
    print(f"Title: {row['document_title']} ({row['char_length']} chars)")
    print(f"Snippet: {row['text'][:200]}...\n")

## 10. Analysis Section 9: Category-wise Document Samples

**Why:** Inspects sample texts across Notifications, Circulars, Orders, Guidelines, and Schemes.

In [ ]:
categories = ['Notification', 'Circular', 'Order', 'Guidelines', 'Scheme', 'Act', 'Scheme Document']
for cat in categories:
    matches = df[df['document_type'].str.contains(cat, case=False, na=False)]
    if not matches.empty:
        sample = matches.iloc[0]
        print(f"Category [{cat}]: Title='{sample['document_title']}'")
        print(f"  Snippet: {str(sample['text'])[:150]}...\n")

## 11. Analysis Section 10: Duplicate Detection

**Why:** Identifies exact duplicate documents to prevent data leakage.

In [ ]:
exact_text_duplicates = df.duplicated(subset=['text']).sum()
title_duplicates = df.duplicated(subset=['document_title']).sum()

print(f"Duplicate Detection Summary:")
print(f" - Exact Duplicate Text Entries: {exact_text_duplicates}")
print(f" - Duplicate Document Titles: {title_duplicates}")

## 12. Export Statistics to dataset_statistics.json

**Why:** Saves structured metrics to `data/processed/dataset_statistics.json`.

In [ ]:
export_stats = {
    "total_documents": int(total_records),
    "document_type_distribution": {str(k): int(v) for k, v in doc_type_counts.items()},
    "jurisdiction_distribution": {str(k): int(v) for k, v in jurisdiction_counts.items()},
    "missing_values": missing_counts,
    "character_length_stats": {k: float(v) for k, v in stats_char.items()},
    "word_length_stats": {k: float(v) for k, v in stats_word.items()},
    "top_issuing_authorities": {str(k): int(v) for k, v in authority_counts.items()},
    "exact_text_duplicates": int(exact_text_duplicates),
    "title_duplicates": int(title_duplicates)
}

os.makedirs(os.path.dirname(STATS_OUTPUT_FILE), exist_ok=True)
with open(STATS_OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(export_stats, f, indent=2)

print(f"Successfully exported dataset statistics to: {STATS_OUTPUT_FILE}")